# LLM으로 지식 그래프 추출하고 Neo4j에 저장

## 환경

### 1) Neo4j

Neo4j Desktop DB 실행

- Neo4j Browser: `http://localhost:7474`
- Bolt: `bolt://localhost:7687`
- 사용자: `neo4j`
- 비밀번호: `graphragpassword`

### 2) `.env`

```env
OPENAI_API_KEY=your_openai_api_key
OPENAI_MODEL=gpt-5.5

NEO4J_URI=bolt://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=graphragpassword
NEO4J_DATABASE=neo4j
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")

print("URI:", URI or "(미설정)")
print("DATABASE:", DATABASE)

## 1. Structured Output 스키마

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class KGNode(BaseModel):
    id: str = Field(description="노드 이름")
    type: Literal[
        "Customer",
        "Order",
        "Product",
        "Warehouse",
        "Courier",
        "Coupon",
        "Unknown",
    ]

class KGRelationship(BaseModel):
    source: str = Field(description="출발 노드 id")
    target: str = Field(description="도착 노드 id")
    kind: Literal[
        "PLACED",
        "CONTAINS",
        "FULFILLED_BY",
        "SHIPPED_BY",
        "USES_COUPON",
        "RELATED_TO",
    ]

class KGGraph(BaseModel):
    nodes: list[KGNode]
    relationships: list[KGRelationship]

## 2. Graph Pruning

관계의 source와 target이 실제 nodes 목록에 모두 존재하는 경우만 남깁니다.

In [ ]:
def validate_kg(kg: KGGraph) -> KGGraph:
   
    return KGGraph()

## 3. 온라인 쇼핑몰 문장

In [ ]:
text = """
고객 A는 주문 O-1001을 생성했고, 주문에는 무선 이어폰이 포함되어 있다.
주문 O-1001에는 노트북 파우치도 포함되어 있다.
주문 O-1001은 이천 물류센터에서 출고되며 빠른택배가 배송한다.
주문 O-1001은 여름할인 쿠폰을 사용했다.

고객 B는 주문 O-1002를 생성했고, 주문에는 스마트 워치가 포함되어 있다.
주문 O-1002는 부산 물류센터에서 출고되며 안심택배가 배송한다.
주문 O-1002는 신규가입 쿠폰을 사용했다.
""".strip()

print(text)

## 4. 추출 프롬프트

관계의 의미와 방향을 예시로 고정합니다.

In [ ]:
prompt = f"""
다음 한국어 문장에서 온라인 쇼핑몰 지식 그래프를 추출하세요.

규칙:
- 문장에 명시된 정보만 사용하세요.
- 같은 의미의 노드는 하나로 합치세요.
- relationship의 source와 target은 반드시 nodes의 id 중 하나여야 합니다.
- 관계 방향은 아래 규칙을 따르세요.

관계 타입:
- PLACED: Customer -> Order
- CONTAINS: Order -> Product
- FULFILLED_BY: Order -> Warehouse
- SHIPPED_BY: Order -> Courier
- USES_COUPON: Order -> Coupon
- RELATED_TO: 위 관계로 표현하기 어려운 일반 관계

문장:
{text}
"""

print(prompt)

## 5. 기본 offline 결과

API 없이도 나머지 적재 코드를 학습할 수 있도록 예상 Structured Output을 준비합니다.

In [ ]:
offline_kg = KGGraph(
    nodes=[
        KGNode(id="고객 A", type="Customer"),
        KGNode(id="고객 B", type="Customer"),
        KGNode(id="주문 O-1001", type="Order"),
        KGNode(id="주문 O-1002", type="Order"),
        KGNode(id="무선 이어폰", type="Product"),
        KGNode(id="스마트 워치", type="Product"),
        KGNode(id="노트북 파우치", type="Product"),
        KGNode(id="이천 물류센터", type="Warehouse"),
        KGNode(id="부산 물류센터", type="Warehouse"),
        KGNode(id="빠른택배", type="Courier"),
        KGNode(id="안심택배", type="Courier"),
        KGNode(id="여름할인 쿠폰", type="Coupon"),
        KGNode(id="신규가입 쿠폰", type="Coupon"),
    ],
    relationships=[
        KGRelationship(
            source="고객 A",
            target="주문 O-1001",
            kind="PLACED",
        ),
        KGRelationship(
            source="고객 B",
            target="주문 O-1002",
            kind="PLACED",
        ),
        KGRelationship(
            source="주문 O-1001",
            target="무선 이어폰",
            kind="CONTAINS",
        ),
        KGRelationship(
            source="주문 O-1001",
            target="노트북 파우치",
            kind="CONTAINS",
        ),
        KGRelationship(
            source="주문 O-1002",
            target="스마트 워치",
            kind="CONTAINS",
        ),
        KGRelationship(
            source="주문 O-1001",
            target="이천 물류센터",
            kind="FULFILLED_BY",
        ),
        KGRelationship(
            source="주문 O-1002",
            target="부산 물류센터",
            kind="FULFILLED_BY",
        ),
        KGRelationship(
            source="주문 O-1001",
            target="빠른택배",
            kind="SHIPPED_BY",
        ),
        KGRelationship(
            source="주문 O-1002",
            target="안심택배",
            kind="SHIPPED_BY",
        ),
        KGRelationship(
            source="주문 O-1001",
            target="여름할인 쿠폰",
            kind="USES_COUPON",
        ),
        KGRelationship(
            source="주문 O-1002",
            target="신규가입 쿠폰",
            kind="USES_COUPON",
        ),
    ],
)

print(offline_kg)

## 6. 선택: OpenAI Structured Output

In [ ]:
import os

from langchain_openai import ChatOpenAI

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY를 설정하세요.")

llm = ChatOpenAI(
   
)
structured_llm = 


kg = 

print("최초 LLM 답변:")
print(kg)


## 7. `validate_kg()` 적용

In [ ]:
kg = 

nodes = 
relationships = [

]

print("nodes:")
for node in nodes:
    print(" ", node)

print("\nrelationships:")
for relationship in relationships:
    print(" ", relationship)

## 8. 검증 동작 실험

존재하지 않는 노드를 가리키는 관계가 제거되는지 확인합니다.

In [ ]:
bad_kg = KGGraph(
    nodes=offline_kg.nodes,
    relationships=[
        *offline_kg.relationships,
        KGRelationship(
            source="주문 O-1001",
            target="존재하지 않는 택배사",
            kind="SHIPPED_BY",
        ),
    ],
)

pruned = validate_kg(bad_kg)
print("검증 전 관계:", len(bad_kg.relationships))
print("검증 후 관계:", len(pruned.relationships))

## 9. Neo4j 저장 Cypher

In [ ]:
def node_cypher(node_type: str) -> str:
    return f"""
    UNWIND $nodes AS node
    WITH node
    WHERE node.type = $node_type
    MERGE (entity:{node_type} {{id: node.id}})
    SET entity.name = node.id,
        entity.type = node.type
    """.strip()


def relationship_cypher(kind: str) -> str:
    return f"""
    UNWIND $relationships AS relationship
    WITH relationship
    WHERE relationship.kind = $kind
    MATCH (source {{id: relationship.source}})
    MATCH (target {{id: relationship.target}})
    MERGE (source)-[:{kind}]->(target)
    """.strip()


print(node_cypher("Product"))
print("\n관계 예시:\n", relationship_cypher("SHIPPED_BY"))

## 10. 선택: Neo4j 적재

03 노트북과 동일한 노드 ID·라벨·관계명을 사용하므로 기존 노드와 자연스럽게 합쳐집니다.

In [ ]:
RUN_LIVE_WRITE = False

if RUN_LIVE_WRITE:
    from langchain_neo4j import Neo4jGraph

    URI = os.getenv("NEO4J_URI")
    USERNAME = os.getenv("NEO4J_USERNAME")
    PASSWORD = os.getenv("NEO4J_PASSWORD")
    DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

    if not all([URI, USERNAME, PASSWORD]):
        raise RuntimeError(".env에 Neo4j 접속 정보를 설정하세요.")

    graph = Neo4jGraph(
        url=URI,
        username=USERNAME,
        password=PASSWORD,
        database=DATABASE,
    )

    node_types = [
        "Customer",
        "Order",
        "Product",
        "Warehouse",
        "Courier",
        "Coupon",
        "Unknown",
    ]

    graph.query("""
    MATCH (entity:Entity)
    WHERE entity.type IN [
        "Customer", "Order", "Product", "Warehouse",
        "Courier", "Coupon", "Unknown"
    ]
    FOREACH (_ IN CASE WHEN entity.type = "Customer" THEN [1] ELSE [] END |
        SET entity:Customer)
    FOREACH (_ IN CASE WHEN entity.type = "Order" THEN [1] ELSE [] END |
        SET entity:Order)
    FOREACH (_ IN CASE WHEN entity.type = "Product" THEN [1] ELSE [] END |
        SET entity:Product)
    FOREACH (_ IN CASE WHEN entity.type = "Warehouse" THEN [1] ELSE [] END |
        SET entity:Warehouse)
    FOREACH (_ IN CASE WHEN entity.type = "Courier" THEN [1] ELSE [] END |
        SET entity:Courier)
    FOREACH (_ IN CASE WHEN entity.type = "Coupon" THEN [1] ELSE [] END |
        SET entity:Coupon)
    FOREACH (_ IN CASE WHEN entity.type = "Unknown" THEN [1] ELSE [] END |
        SET entity:Unknown)
    REMOVE entity:Entity
    """)

    for node_type in node_types:
        graph.query(
            f"""
            CREATE CONSTRAINT {node_type.lower()}_id_unique IF NOT EXISTS
            FOR (entity:{node_type})
            REQUIRE entity.id IS UNIQUE
            """,
        )
        graph.query(
            node_cypher(node_type),
            params={
                "nodes": nodes,
                "node_type": node_type,
            },
        )

    relationship_types = [
        "PLACED",
        "CONTAINS",
        "FULFILLED_BY",
        "SHIPPED_BY",
        "USES_COUPON",
        "RELATED_TO",
    ]
    for kind in relationship_types:
        graph.query(
            relationship_cypher(kind),
            params={
                "relationships": relationships,
                "kind": kind,
            },
        )

    graph.refresh_schema()
    print("그래프 생성 완료!")
    print(graph.schema)
else:
    print("확인 모드: Neo4j에 저장하지 않았습니다.")